# Perfil Económico de Municipios de Morelos por Tipo de Actividad

**Pregunta guía:** ¿Qué municipios de Morelos tienen perfiles económicos parecidos según sus unidades económicas?

**Fuente de datos:** DENUE (Directorio Estadístico Nacional de Unidades Económicas) - INEGI  
**URL:** https://www.inegi.org.mx/app/descarga/?ti=6  

**Modelo sugerido:** KMeans (aprendizaje no supervisado)

---
## 1. Carga de librerías y dataset

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/raw/dataset_original.csv', encoding='latin-1', low_memory=False)

In [ ]:
df.head()

---
## 2. Revisión inicial del dataset

In [ ]:
# Dimensiones del dataset (filas, columnas)
print('Shape del dataset:', df.shape)

In [ ]:
# Nombres de todas las columnas
print('Columnas del dataset:')
print(list(df.columns))

In [ ]:
# Tipos de datos por columna
df.dtypes

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Estadísticas descriptivas de columnas numéricas
df.describe()

---
## 3. Revisión de duplicados

In [ ]:
# Contar filas duplicadas
duplicados = df.duplicated().sum()
print('Filas duplicadas:', duplicados)

In [ ]:
# Si hay duplicados, eliminarlos
if duplicados > 0:
    df = df.drop_duplicates()
    print('Duplicados eliminados. Nuevo shape:', df.shape)
else:
    print('No se encontraron duplicados. No se requiere acción.')

---
## 4. Revisión de valores nulos

In [ ]:
# Total de nulos por columna
nulos = df.isnull().sum()
print('Nulos por columna:')
print(nulos)

In [ ]:
# Porcentaje de nulos por columna (solo las que tienen nulos)
porcentaje_nulos = (df.isnull().sum() / len(df) * 100).round(2)
nulos_filtrados = porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)
print('Porcentaje de nulos (solo columnas con nulos):')
print(nulos_filtrados)

**Interpretación:** Muchas columnas tienen altos porcentajes de nulos (edificio, numero_int, correoelec, www, etc.) porque son campos opcionales en el registro del DENUE. Esto no representa un error en los datos, sino la naturaleza del registro. Las columnas clave para el análisis (`municipio`, `nombre_act`, `per_ocu`, `codigo_act`, `entidad`) no tienen nulos.

---
## 5. Revisión de valores únicos en columnas clave

In [ ]:
# Verificar que solo tenemos datos de Morelos
print('Entidades únicas:', df['entidad'].nunique())
print(df['entidad'].unique())

In [ ]:
# Municipios únicos
print('Municipios únicos:', df['municipio'].nunique())
print(sorted(df['municipio'].unique()))

In [ ]:
# Top 10 actividades económicas más frecuentes
print('Top 10 actividades económicas:')
print(df['nombre_act'].value_counts().head(10))

In [ ]:
# Distribución de personal ocupado
print('Distribución de personal ocupado:')
print(df['per_ocu'].value_counts())

In [ ]:
# Tipos de unidad económica
print('Tipos de unidad económica:')
print(df['tipoUniEco'].value_counts())

---
## 6. Limpieza básica

Para el análisis de perfil económico por municipio, no necesitamos todas las 42 columnas. Vamos a:
1. Seleccionar solo las columnas relevantes para el análisis.
2. Crear una variable de sector económico a partir del código de actividad.
3. Verificar que no haya valores inconsistentes en las columnas seleccionadas.

In [ ]:
# Seleccionar columnas relevantes para el análisis
columnas_relevantes = [
    'id',           # Identificador único de la unidad económica
    'codigo_act',   # Código de actividad económica (SCIAN)
    'nombre_act',   # Nombre de la actividad económica
    'per_ocu',      # Rango de personal ocupado
    'cve_mun',      # Clave del municipio
    'municipio',    # Nombre del municipio
    'entidad',      # Nombre de la entidad
    'tipoUniEco',   # Tipo de unidad económica
    'latitud',      # Latitud
    'longitud'      # Longitud
]

df_sel = df[columnas_relevantes].copy()
print('Shape después de seleccionar columnas:', df_sel.shape)
df_sel.head()

In [ ]:
# Crear variable de sector económico a partir de los 2 primeros dígitos del código SCIAN
df_sel['sector_codigo'] = df_sel['codigo_act'].astype(str).str[:2]

# Mapear códigos de sector a nombres descriptivos según el SCIAN
sectores_scian = {
    '11': 'Agricultura y ganadería',
    '21': 'Minería',
    '22': 'Electricidad y agua',
    '23': 'Construcción',
    '31': 'Manufactura',
    '32': 'Manufactura',
    '33': 'Manufactura',
    '43': 'Comercio al por mayor',
    '46': 'Comercio al por menor',
    '48': 'Transporte',
    '49': 'Transporte',
    '51': 'Información y medios',
    '52': 'Servicios financieros',
    '53': 'Servicios inmobiliarios',
    '54': 'Servicios profesionales',
    '55': 'Corporativos',
    '56': 'Servicios de apoyo',
    '61': 'Servicios educativos',
    '62': 'Servicios de salud',
    '71': 'Esparcimiento y cultura',
    '72': 'Alojamiento y alimentos',
    '81': 'Otros servicios',
    '93': 'Gobierno'
}

df_sel['sector_nombre'] = df_sel['sector_codigo'].map(sectores_scian)
print('Sectores asignados:')
print(df_sel['sector_nombre'].value_counts())

In [ ]:
# Verificar si hay sectores no mapeados (NaN)
sin_sector = df_sel['sector_nombre'].isnull().sum()
print('Unidades sin sector asignado:', sin_sector)

if sin_sector > 0:
    print('Códigos de sector no mapeados:')
    print(df_sel[df_sel['sector_nombre'].isnull()]['sector_codigo'].unique())

In [ ]:
# Eliminar filas sin sector asignado 
if sin_sector > 0:
    df_sel = df_sel.dropna(subset=['sector_nombre'])
    print('Filas eliminadas sin sector. Nuevo shape:', df_sel.shape)
else:
    print('Todas las filas tienen sector asignado. No se requiere acción.')

In [ ]:
# Verificar nulos en las columnas seleccionadas
print('Nulos en columnas seleccionadas:')
print(df_sel.isnull().sum())

In [ ]:
# Verificar duplicados por ID de unidad económica
duplicados_id = df_sel.duplicated(subset=['id']).sum()
print('Duplicados por ID:', duplicados_id)

if duplicados_id > 0:
    df_sel = df_sel.drop_duplicates(subset=['id'])
    print('Duplicados por ID eliminados. Nuevo shape:', df_sel.shape)
else:
    print('No hay duplicados por ID.')

In [ ]:
# Resetear índice después de limpieza
df_sel = df_sel.reset_index(drop=True)
print('Shape final del dataset limpio:', df_sel.shape)
df_sel.head()

---
## 7. Resumen de la limpieza realizada

**Resumen de la limpieza:**

1. **Dataset original:** 113,066 filas × 42 columnas del DENUE (INEGI) para el estado de Morelos.
2. **Duplicados:** Se verificaron y no se encontraron filas completamente duplicadas.
3. **Nulos:** Se identificaron columnas con altos porcentajes de nulos (campos opcionales de dirección y contacto). Las columnas clave para el análisis no tienen nulos.
4. **Selección de columnas:** Se seleccionaron 10 columnas relevantes para el análisis de perfil económico.
5. **Creación de variable sector:** Se creó la variable `sector_nombre` a partir de los primeros 2 dígitos del código SCIAN.
6. **Eliminación de registros incompletos:** Se eliminaron filas sin sector económico asignado (si las hubo).
7. **Verificación final de duplicados por ID.**

---
## 8. Exportar dataset limpio

In [ ]:
# Guardar dataset limpio
df_sel.to_csv('../data/processed/dataset_limpio.csv', index=False, encoding='utf-8')
print('Dataset limpio guardado en: data/processed/dataset_limpio.csv')
print('Shape final:', df_sel.shape)